# Task 3: ASR with Custom Recordings

In this task, we will:
1. Use the **Whisper** ASR model.
2. Record specific sentences.
3. Run and evaluate the ASR model on these recordings.

In [1]:
%pip install jiwer
%pip install librosa

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import pipeline
from jiwer import wer
import librosa
import numpy as np
import IPython.display as ipd
import os

/Users/kaungkhantlin/Developer/2_2025/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Load the ASR Model (Whisper)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# Using openai/whisper-small as a balance between speed and accuracy
asr_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=device)
print(f"Model loaded on {device}")

Device set to use cpu


Model loaded on cpu


### 2. Define Sentences to Record

Please record the following sentences and save them as `.wav` files. You can place them in a folder (e.g., `my_recordings/`) or in the same directory.
For this notebook, we assume the file names are `s1.wav`, `s2.wav`, etc.

In [9]:
sentences = {
    "s1": "I just came back from the bank",
    "s2": "We love to play tennis",
    "s3": "I can’t tell whether the weather will affect where we’re going",
    "s4": "Dr. Shrestha emailed vmes@au.edu on December 3rd, 2025 at 11:59 p.m.",
    "s5": "The API uses HTTPS, JSON, and OAuth 2.0",
    "s6": "The transformer’s self-attention mechanism mitigates vanishing gradients during backpropagation",
    "s7": "I was born in—uh—no, actually I grew up in—wait—Chiang Mai.",
    "s8": "I’ll deploy the model after lunch, ခုနနေရင် ပြန်တွေ့မယ်"
}

# Ensure the output directory exists
os.makedirs("myRecordings", exist_ok=True)
print("Please save your recordings as myRecordings/s1.wav, myRecordings/s2.wav, etc.")

Please save your recordings as myRecordings/s1.wav, myRecordings/s2.wav, etc.


### 3. (Optional) Record Audio directly
If you have `sounddevice` and `scipy` installed, you can use the code below to record directly. Otherwise, verify your files are in `my_recordings/` folder.

In [5]:
# Helper function to record audio (requires sounddevice)
# Uncomment to use if you can record on this machine

# import sounddevice as sd
# from scipy.io.wavfile import write

# def record_sentence(key, text, duration=5):
#     print(f"Prepare to read: '{text}'")
#     print(f"Recording to my_recordings/{key}.wav in 3 seconds...")
#     input("Press Enter to start recording...")
#     print("Recording...")
#     fs = 16000  # Sample rate
#     myrecording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
#     sd.wait()  # Wait until recording is finished
#     write(f"my_recordings/{key}.wav", fs, myrecording)
#     print(f"Saved specific file: my_recordings/{key}.wav\n")

# # Example usage:
# # record_sentence("s1", sentences["s1"], duration=4)

### 4. Run ASR and Evaluate
We will iterate through the sentences, transcribe the audio, and calculate the Word Error Rate (WER).

In [10]:
import warnings
warnings.filterwarnings("ignore")

for key, reference_text in sentences.items():
    file_path = f"myRecordings/{key}.wav"
    
    if not os.path.exists(file_path):
        print(f"Skipping {key}: File {file_path} not found.")
        continue
    
    print(f"--- Analyzing {key} ---")
    
    # Load audio
    # librosa loads with floating point, and we can set sr=16000 directly
    audio_array, sampling_rate = librosa.load(file_path, sr=16000)
    
    # Play current audio (optional visual check in notebook)
    # ipd.display(ipd.Audio(audio_array, rate=sampling_rate)) # Uncomment to show audio player
    
    # Transcribe
    # If the sentence is multilingual instructions (s8), Whisper handles it automatically
    prediction = asr_pipeline(audio_array)
    predicted_text = prediction["text"]
    
    # Evaluate
    # Basic normalization (lowercase, remove punctuation roughly if needed, but wer handles basic matching)
    error_rate = wer(reference_text.lower(), predicted_text.lower())
    
    print(f"Reference: {reference_text}")
    print(f"Predicted: {predicted_text}")
    print(f"WER: {error_rate:.4f}\n")

--- Analyzing s1 ---
Reference: I just came back from the bank
Predicted:  I just came back from the bang.
WER: 0.1429

--- Analyzing s2 ---
Reference: We love to play tennis
Predicted:  We love to play tennis.
WER: 0.2000

--- Analyzing s3 ---
Reference: I can’t tell whether the weather will affect where we’re going
Predicted:  I can tell whether the weather will affect where we are going.
WER: 0.3636

--- Analyzing s4 ---
Reference: Dr. Shrestha emailed vmes@au.edu on December 3rd, 2025 at 11:59 p.m.
Predicted:  Dr. Sharista E-Mail, V-M-E-S, at AU.edu, on December 3, 2025, at 11.59 p.m.
WER: 0.7273

--- Analyzing s5 ---
Reference: The API uses HTTPS, JSON, and OAuth 2.0
Predicted: The API uses HTTPS.json and OAUTH2.0
WER: 0.5000

--- Analyzing s6 ---
Reference: The transformer’s self-attention mechanism mitigates vanishing gradients during backpropagation
Predicted:  The transformer's self-attention mechanism mitigates vanishing gradients during back propagation.
WER: 0.3333

--- Ana